In [14]:
"""
Quy trình chọn lớp cho bài toán nhận diện sketch sớm (Quick, Draw!).

  Bước 1  stats    : quét toàn bộ lớp, tính thống kê từng lớp (lấy mẫu ngẫu nhiên)
                     + ảnh trung bình của phần MỞ ĐẦU hình vẽ (25% số nét đầu)
  Bước 2  propose  : gom cụm các lớp có phần mở đầu giống nhau (dễ nhầm khi đoán sớm),
                     chia bậc độ khó theo tỉ lệ recognized, đề xuất ~100 lớp
  Bước 3  (nhóm)   : mở proposal.csv, sửa cột keep (1/0), thêm lớp muốn thêm
  Bước 4  freeze   : chốt danh sách -> classes.txt (không cho ghi đè nếu không --force)

  "all" = stats + propose chạy liền.

Dòng lệnh:
  python class_selection.py all --source kaggle \
      --root /kaggle/input/quickdraw-doodle-recognition \
      --include "The Mona Lisa,camouflage,yoga"
  python class_selection.py freeze

Trong notebook:
  main(["all", "--source", "kaggle", "--root", "/kaggle/input/quickdraw-doodle-recognition"])
  main(["freeze"])

Nguồn dữ liệu:
  --source kaggle : thư mục cuộc thi Kaggle (ROOT/train_simplified/*.csv)
  --source ndjson : thư mục chứa các file simplified/*.ndjson đã tải từ Google

Mọi thống kê tính TRƯỚC khi train mô hình nào, nên dùng được để định nghĩa
tập khó "trước khi xem kết quả" như spec yêu cầu.
"""

import argparse
import datetime
import hashlib
import json
import math
import os
import random
import sys
import time

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

IMG = 32  # kích thước ảnh trung bình dùng để so sánh hình dạng


# =====================================================================
# Đọc dữ liệu
# =====================================================================
def list_classes(source, root):
    if source == "kaggle":
        folder, ext = os.path.join(root, "train_simplified"), ".csv"
    else:
        folder, ext = root, ".ndjson"
    if not os.path.isdir(folder):
        sys.exit(f"Không thấy thư mục {folder}. Kiểm tra lại --root "
                 f"(trên Kaggle chạy: !ls /kaggle/input).")
    names = sorted(f[: -len(ext)] for f in os.listdir(folder) if f.endswith(ext))
    if not names:
        sys.exit(f"Không có file {ext} nào trong {folder}.")
    return folder, ext, names


def read_sample(path, ext, frac, rng):
    """Đọc 1 lượt: lấy mẫu ngẫu nhiên theo tỉ lệ frac và đếm tổng số hình."""
    if ext == ".csv":
        state = {"n": 0}

        def skip(i):
            if i == 0:
                return False  # dòng tiêu đề
            state["n"] = i
            return rng.random() > frac

        df = pd.read_csv(path, usecols=["countrycode", "recognized", "drawing"],
                         skiprows=skip)
        df["recognized"] = df["recognized"].astype(str).str.lower() == "true"
        df["drawing"] = df["drawing"].map(json.loads)
        return df, state["n"]

    rows, n = [], 0
    with open(path) as f:
        for line in f:
            n += 1
            if rng.random() <= frac:
                d = json.loads(line)
                rows.append({"countrycode": d.get("countrycode", ""),
                             "recognized": bool(d["recognized"]),
                             "drawing": d["drawing"]})
    return pd.DataFrame(rows, columns=["countrycode", "recognized", "drawing"]), n


# =====================================================================
# Vẽ hình (dùng để tạo ảnh trung bình phần mở đầu / toàn bộ)
# =====================================================================
def prefix(strokes, frac):
    k = max(1, math.ceil(frac * len(strokes)))
    return strokes[:k]


def render(strokes, size=IMG, pad=2):
    """Vẽ các nét vào ảnh size x size, chuẩn hoá theo khung bao của chính các nét đó
    (giống cách mô hình sẽ thấy một hình vẽ dở dang)."""
    xs = [x for s in strokes for x in s[0]]
    ys = [y for s in strokes for y in s[1]]
    img = Image.new("L", (size, size), 0)
    if not xs:
        return np.zeros((size, size), np.float32)
    x0, y0 = min(xs), min(ys)
    w, h = max(xs) - x0, max(ys) - y0
    scale = (size - 2 * pad) / max(w, h, 1)
    ox = pad + ((size - 2 * pad) - w * scale) / 2
    oy = pad + ((size - 2 * pad) - h * scale) / 2
    d = ImageDraw.Draw(img)
    for s in strokes:
        pts = [(ox + (x - x0) * scale, oy + (y - y0) * scale) for x, y in zip(s[0], s[1])]
        if len(pts) == 1:
            d.point(pts, fill=255)
        else:
            d.line(pts, fill=255, width=1)
    return np.asarray(img, np.float32) / 255.0


# =====================================================================
# Bước 1: thống kê
# =====================================================================
def cmd_stats(a):
    folder, ext, names = list_classes(a.source, a.root)
    if a.limit:
        names = names[: a.limit]
    os.makedirs(a.out_dir, exist_ok=True)
    rng = random.Random(a.seed)
    print(f"Quét {len(names)} lớp trong {folder} "
          f"(lấy mẫu {a.sample_frac:.0%}, tối đa {a.max_sample} hình/lớp)")

    rows, early_means, full_means = [], [], []
    t0 = time.time()
    for i, name in enumerate(names, 1):
        df, n_total = read_sample(os.path.join(folder, name + ext), ext, a.sample_frac, rng)
        if len(df) > a.max_sample:
            df = df.sample(n=a.max_sample, random_state=a.seed)
        if len(df) == 0:
            print(f"  ! {name}: không lấy được mẫu nào, bỏ qua")
            continue
        n_strokes = df["drawing"].map(len)
        n_points = df["drawing"].map(lambda d: sum(len(s[0]) for s in d))
        cc = df["countrycode"].fillna("").astype(str)
        top_cc = cc.value_counts(normalize=True)

        shape = df["drawing"].sample(n=min(a.n_shape, len(df)), random_state=a.seed)
        early_means.append(np.mean([render(prefix(s, a.prefix_frac)) for s in shape], 0))
        full_means.append(np.mean([render(s) for s in shape], 0))

        rows.append({
            "class": name,
            "n_drawings": n_total,
            "n_sampled": len(df),
            "recognized_rate": round(float(df["recognized"].mean()), 4),
            "mean_strokes": round(float(n_strokes.mean()), 2),
            "median_strokes": float(n_strokes.median()),
            "median_points": float(n_points.median()),
            "n_countries": int(cc.nunique()),
            "top_country": top_cc.index[0],
            "top_country_share": round(float(top_cc.iloc[0]), 3),
        })
        if i % 10 == 0 or i == len(names):
            el = time.time() - t0
            print(f"  [{i}/{len(names)}] {el/60:.1f} phút, "
                  f"còn khoảng {el / i * (len(names) - i) / 60:.1f} phút")

    stats = pd.DataFrame(rows)
    stats.to_csv(os.path.join(a.out_dir, "class_stats.csv"), index=False)
    np.savez_compressed(os.path.join(a.out_dir, "shapes.npz"),
                        names=np.array(stats["class"]),
                        early=np.stack(early_means), full=np.stack(full_means),
                        prefix_frac=a.prefix_frac)
    with open(os.path.join(a.out_dir, "stats_config.json"), "w") as f:
        json.dump(vars(a), f, indent=2, ensure_ascii=False)
    print(f"\nĐã lưu thống kê {len(stats)} lớp vào {a.out_dir}/class_stats.csv")


# =====================================================================
# Bước 2: gom cụm + đề xuất
# =====================================================================
def _cosine(m):
    v = m.reshape(len(m), -1).astype(np.float64)
    v -= v.mean(0)  # trừ hình trung bình chung để nổi bật khác biệt giữa các lớp
    v /= np.linalg.norm(v, axis=1, keepdims=True) + 1e-9
    return v @ v.T


def _split_list(text, file):
    out = []
    if text:
        out += [c.strip() for c in text.split(",") if c.strip()]
    if file:
        with open(file) as f:
            out += [l.strip() for l in f if l.strip()]
    return out


def cmd_propose(a):
    from scipy.cluster.hierarchy import fcluster, linkage
    from scipy.spatial.distance import squareform

    stats_path = os.path.join(a.out_dir, "class_stats.csv")
    if not os.path.exists(stats_path):
        sys.exit("Chưa có class_stats.csv, chạy bước stats trước.")
    stats = pd.read_csv(stats_path)
    sh = np.load(os.path.join(a.out_dir, "shapes.npz"), allow_pickle=True)
    names = list(sh["names"])
    stats = stats.set_index("class").loc[names].reset_index()
    C = len(names)

    # --- độ giống nhau của phần mở đầu và của toàn bộ hình ---
    S_early, S_full = _cosine(sh["early"]), _cosine(sh["full"])
    np.fill_diagonal(S_early, -1)
    nearest = S_early.argmax(1)
    stats["early_confusability"] = S_early.max(1).round(4)
    stats["nearest_early"] = [names[j] for j in nearest]
    stats["full_sim_to_nearest"] = [round(S_full[i, j], 4) for i, j in enumerate(nearest)]
    # cao = đầu giống, cuối khác -> đúng loại khó mà bài toán đoán sớm cần
    stats["early_late_gap"] = (stats["early_confusability"] - stats["full_sim_to_nearest"]).round(4)

    # --- bậc độ khó theo tỉ lệ recognized (thấp = khó) ---
    stats["tier"] = pd.qcut(stats["recognized_rate"].rank(method="first"), 3,
                            labels=["hard", "medium", "easy"]).astype(str)

    # --- gom cụm theo hình dạng mở đầu ---
    n_clusters = min(a.n_clusters, C)
    D = np.clip(1 - S_early, 0, 2)
    np.fill_diagonal(D, 0)
    Z = linkage(squareform(D, checks=False), method="average")
    stats["cluster"] = fcluster(Z, n_clusters, criterion="maxclust")

    tight = {}
    for cl, g in stats.groupby("cluster"):
        idx = g.index.to_numpy()
        tight[cl] = float(S_early[np.ix_(idx, idx)][~np.eye(len(idx), dtype=bool)].mean()) \
            if len(idx) > 1 else 0.0
    stats["cluster_tightness"] = stats["cluster"].map(tight).round(4)
    stats["cluster_size"] = stats["cluster"].map(stats["cluster"].value_counts())

    # --- chọn lớp ---
    include = _split_list(a.include, a.include_file)
    exclude = set(_split_list(a.exclude, a.exclude_file))
    unknown = [c for c in include + list(exclude) if c not in names]
    if unknown:
        sys.exit(f"Không có các lớp: {unknown}. Xem tên đúng trong class_stats.csv.")

    role, reason = {}, {}
    for c in include:
        role[c], reason[c] = "included", "nhóm chọn thủ công"
    n_target = a.n_classes
    if len(role) > n_target:
        print(f"! Số lớp --include ({len(role)}) vượt --n-classes ({n_target})")

    conf_target = round(n_target * a.confusable_share)
    n_conf = 0
    order = sorted((cl for cl in tight if stats["cluster_size"][stats["cluster"] == cl].iloc[0]
                    >= a.min_cluster_size), key=lambda cl: -tight[cl])
    for cl in order:
        if n_conf >= conf_target or len(role) >= n_target:
            break
        g = stats[(stats["cluster"] == cl) & ~stats["class"].isin(exclude)]
        # loại thành viên "lạc": giống trung bình với các lớp còn lại trong cụm quá thấp
        idx = g.index.to_numpy()
        if len(idx) > 1:
            sub = S_early[np.ix_(idx, idx)]
            member_sim = (sub.sum(1) - np.diag(sub)) / (len(idx) - 1)
            g = g[member_sim >= a.member_ratio * tight[cl]]
        if len(g) < 2:
            continue
        g = g.sort_values("early_late_gap", ascending=False).head(a.per_cluster)
        members = ", ".join(g["class"].head(4))
        for c in g["class"]:
            if c in role or len(role) >= n_target:
                continue
            role[c] = "confusable"
            reason[c] = f"cụm {cl} (độ giống mở đầu {tight[cl]:.2f}): {members}…"
            n_conf += 1

    # đối chứng: phần mở đầu khác biệt nhất, chia đều theo bậc khó
    median_conf = stats["early_confusability"].median()
    pool = stats[~stats["class"].isin(list(role) + list(exclude))]
    pool = pool.sort_values("early_confusability")
    tiers = ["hard", "medium", "easy"]
    for low_only in (True, False):  # ưu tiên lớp dưới trung vị, thiếu mới lấy thêm
        cand = pool[~pool["class"].isin(list(role))]
        if low_only:
            cand = cand[cand["early_confusability"] <= median_conf]
        by_tier = {t: list(g["class"]) for t, g in cand.groupby("tier")}
        k = 0
        while len(role) < n_target and any(by_tier.get(t) for t in tiers):
            t = tiers[k % 3]
            if by_tier.get(t):
                c = by_tier[t].pop(0)
                v = float(stats.loc[stats["class"] == c, "early_confusability"].iloc[0])
                role[c] = "control" if low_only else "filler"
                reason[c] = (f"đối chứng: mở đầu ít giống lớp khác ({v:.2f}, bậc {t})"
                             if low_only else
                             f"bổ sung cho đủ số lớp ({v:.2f}, bậc {t}), nên xem lại")
            k += 1

    stats["role"] = stats["class"].map(role).fillna("")
    stats["reason"] = stats["class"].map(reason).fillna("")
    stats["keep"] = (stats["role"] != "").astype(int)

    cols = ["keep", "class", "role", "reason", "cluster", "tier", "recognized_rate",
            "mean_strokes", "early_confusability", "nearest_early", "early_late_gap",
            "n_drawings", "n_countries"]
    prop = stats[stats["keep"] == 1].sort_values(["role", "cluster", "class"])[cols]
    prop.to_csv(os.path.join(a.out_dir, "proposal.csv"), index=False)
    stats.sort_values(["cluster", "class"]).to_csv(
        os.path.join(a.out_dir, "all_classes_ranked.csv"), index=False)

    _plot_clusters(stats, sh["early"], sh["full"], names, a.out_dir)

    # --- tóm tắt ---
    print(f"Đề xuất {len(prop)} lớp -> {a.out_dir}/proposal.csv")
    print("  Theo vai trò:", prop["role"].value_counts().to_dict())
    tier_share = prop["tier"].value_counts(normalize=True)
    print("  Theo bậc khó:", {t: f"{v:.0%}" for t, v in tier_share.items()})
    if tier_share.get("hard", 0) < 0.2:
        print("  ! Bậc 'hard' dưới 20%: kết quả trên tập khó sẽ dao động nhiều.")
    print(f"  Toàn bộ lớp kèm cụm: {a.out_dir}/all_classes_ranked.csv")
    print(f"  Hình các cụm: {a.out_dir}/clusters.png")
    print("\nTiếp theo: cả nhóm mở proposal.csv, sửa cột keep (1 giữ / 0 bỏ), "
          "có thể thêm dòng mới chỉ cần cột class; rồi chạy bước freeze.")


def _plot_clusters(stats, early, full, names, out_dir, max_rows=15, max_cols=8):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    sel = stats[stats["role"] == "confusable"]
    clusters = (sel.groupby("cluster")["cluster_tightness"].first()
                .sort_values(ascending=False).index[:max_rows])
    if len(clusters) == 0:
        return
    idx_of = {n: i for i, n in enumerate(names)}
    max_cols = max(2, min(max_cols, int(sel[sel["cluster"].isin(clusters)]
                                         .groupby("cluster").size().max())))
    fig, axes = plt.subplots(len(clusters) * 2, max_cols,
                             figsize=(max_cols * 1.4, len(clusters) * 3.0), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for r, cl in enumerate(clusters):
        members = list(sel[sel["cluster"] == cl]["class"])[:max_cols]
        for col, c in enumerate(members):
            i = idx_of[c]
            axes[2 * r, col].imshow(early[i], cmap="gray_r")
            axes[2 * r, col].set_title(c[:14], fontsize=7)
            axes[2 * r + 1, col].imshow(full[i], cmap="gray_r")
        axes[2 * r, 0].text(-4, IMG / 2, f"cụm {cl}\nmở đầu", fontsize=7,
                            ha="right", va="center")
        axes[2 * r + 1, 0].text(-4, IMG / 2, "toàn bộ", fontsize=7, ha="right", va="center")
    fig.suptitle("Ảnh trung bình: hàng trên = phần mở đầu, hàng dưới = hình hoàn chỉnh",
                 fontsize=9)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "clusters.png"), dpi=130)
    plt.close(fig)


# =====================================================================
# Bước 4: chốt danh sách
# =====================================================================
def cmd_freeze(a):
    prop_path = a.proposal or os.path.join(a.out_dir, "proposal.csv")
    stats = pd.read_csv(os.path.join(a.out_dir, "all_classes_ranked.csv"))
    prop = pd.read_csv(prop_path)
    prop["class"] = prop["class"].astype(str).str.strip()
    if "keep" in prop:
        prop["keep"] = pd.to_numeric(prop["keep"], errors="coerce").fillna(1).astype(int)
        prop = prop[prop["keep"] == 1]
    classes = list(dict.fromkeys(prop["class"]))  # bỏ trùng, giữ thứ tự

    unknown = [c for c in classes if c not in set(stats["class"])]
    if unknown:
        sys.exit(f"Không có các lớp: {unknown}. Sửa lại tên trong {prop_path}.")

    txt = os.path.join(a.out_dir, "classes.txt")
    if os.path.exists(txt) and not a.force:
        sys.exit(f"{txt} đã được chốt trước đó. Đổi danh sách sau khi đã có kết quả là "
                 f"selective reporting. Nếu CHƯA train gì, chạy lại với --force.")

    classes = sorted(classes)
    content = "\n".join(classes)
    with open(txt, "w") as f:
        f.write(content)

    frozen = stats.set_index("class").loc[classes].reset_index()
    extra = prop.drop_duplicates("class").set_index("class")
    for col in ["role", "reason"]:
        if col in extra:
            frozen[col] = frozen["class"].map(extra[col]).fillna(frozen[col])
    frozen["role"] = frozen["role"].fillna("").replace("", "added_by_team")
    frozen.to_csv(os.path.join(a.out_dir, "classes_frozen.csv"), index=False)

    info = {
        "frozen_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "n_classes": len(classes),
        "sha256_classes_txt": hashlib.sha256(content.encode()).hexdigest(),
        "by_role": frozen["role"].value_counts().to_dict(),
        "by_tier": frozen["tier"].value_counts().to_dict(),
        "note": a.note or "",
    }
    with open(os.path.join(a.out_dir, "freeze_info.json"), "w") as f:
        json.dump(info, f, indent=2, ensure_ascii=False)

    print(f"Đã chốt {len(classes)} lớp -> {txt}")
    print("  Theo vai trò:", info["by_role"])
    print("  Theo bậc khó:", info["by_tier"])
    print(f"  Mã kiểm tra: {info['sha256_classes_txt'][:16]}… (ghi vào báo cáo / README)")
    print("\nBước tiếp: lấy dữ liệu cho các lớp đã chốt, ví dụ:")
    print(f"  python select_subset.py --source kaggle --root <ROOT> "
          f"--class-file {txt} --n-per-class 20000")


# =====================================================================
def main(argv=None):
    if argv is None and "ipykernel" in sys.modules:
        print('Trong notebook, gọi kèm tham số, ví dụ:\n'
              'main(["all", "--source", "kaggle", "--root", '
              '"/kaggle/input/quickdraw-doodle-recognition"])\n'
              'main(["freeze"])')
        return

    p = argparse.ArgumentParser(description="Chọn lớp Quick, Draw! cho bài toán đoán sớm")
    sub = p.add_subparsers(dest="cmd", required=True)

    def add_common(sp):
        sp.add_argument("--out-dir", default="class_selection")

    def add_stats(sp):
        sp.add_argument("--source", choices=["kaggle", "ndjson"], required=True)
        sp.add_argument("--root", required=True)
        sp.add_argument("--sample-frac", type=float, default=0.03,
                        help="tỉ lệ hình lấy mẫu mỗi lớp (mặc định 3%%)")
        sp.add_argument("--max-sample", type=int, default=3000)
        sp.add_argument("--n-shape", type=int, default=400,
                        help="số hình dùng để tạo ảnh trung bình mỗi lớp")
        sp.add_argument("--prefix-frac", type=float, default=0.25,
                        help="phần mở đầu = bao nhiêu %% số nét đầu tiên")
        sp.add_argument("--limit", type=int, default=0, help="chỉ quét N lớp đầu (để debug)")
        sp.add_argument("--seed", type=int, default=42)

    def add_propose(sp):
        sp.add_argument("--n-classes", type=int, default=100)
        sp.add_argument("--n-clusters", type=int, default=40)
        sp.add_argument("--per-cluster", type=int, default=8)
        sp.add_argument("--min-cluster-size", type=int, default=3)
        sp.add_argument("--member-ratio", type=float, default=0.75,
                        help="loại thành viên có độ giống TB < tỉ lệ này x độ chặt của cụm")
        sp.add_argument("--confusable-share", type=float, default=0.7,
                        help="tỉ lệ lớp thuộc cụm dễ nhầm (phần còn lại là đối chứng)")
        sp.add_argument("--include", help='lớp bắt buộc có, vd "The Mona Lisa,camouflage"')
        sp.add_argument("--include-file")
        sp.add_argument("--exclude", help="lớp loại bỏ")
        sp.add_argument("--exclude-file")

    sp = sub.add_parser("stats"); add_common(sp); add_stats(sp)
    sp = sub.add_parser("propose"); add_common(sp); add_propose(sp)
    sp = sub.add_parser("all"); add_common(sp); add_stats(sp); add_propose(sp)
    sp = sub.add_parser("freeze"); add_common(sp)
    sp.add_argument("--proposal", help="đường dẫn proposal.csv đã sửa (mặc định trong out-dir)")
    sp.add_argument("--note", help="ghi chú lý do chọn, lưu vào freeze_info.json")
    sp.add_argument("--force", action="store_true")

    a = p.parse_args(argv)
    if a.cmd in ("stats", "all"):
        cmd_stats(a)
    if a.cmd in ("propose", "all"):
        cmd_propose(a)
    if a.cmd == "freeze":
        cmd_freeze(a)
    # main(["freeze", "--note", "lý do chọn lớp của nhóm"])

In [15]:
main(["all", "--source", "kaggle",
  "--root", "/kaggle/input/competitions/quickdraw-doodle-recognition",
  "--include", "The Mona Lisa,camouflage,yoga"])

Quét 340 lớp trong /kaggle/input/competitions/quickdraw-doodle-recognition/train_simplified (lấy mẫu 3%, tối đa 3000 hình/lớp)
  [10/340] 0.1 phút, còn khoảng 3.0 phút
  [20/340] 0.2 phút, còn khoảng 2.7 phút
  [30/340] 0.3 phút, còn khoảng 2.6 phút
  [40/340] 0.3 phút, còn khoảng 2.6 phút
  [50/340] 0.4 phút, còn khoảng 2.5 phút
  [60/340] 0.5 phút, còn khoảng 2.5 phút
  [70/340] 0.6 phút, còn khoảng 2.4 phút
  [80/340] 0.7 phút, còn khoảng 2.2 phút
  [90/340] 0.8 phút, còn khoảng 2.2 phút
  [100/340] 0.9 phút, còn khoảng 2.1 phút
  [110/340] 0.9 phút, còn khoảng 1.9 phút
  [120/340] 1.0 phút, còn khoảng 1.9 phút
  [130/340] 1.1 phút, còn khoảng 1.8 phút
  [140/340] 1.3 phút, còn khoảng 1.8 phút
  [150/340] 1.3 phút, còn khoảng 1.7 phút
  [160/340] 1.4 phút, còn khoảng 1.6 phút
  [170/340] 1.5 phút, còn khoảng 1.5 phút
  [180/340] 1.6 phút, còn khoảng 1.4 phút
  [190/340] 1.7 phút, còn khoảng 1.3 phút
  [200/340] 1.7 phút, còn khoảng 1.2 phút
  [210/340] 1.8 phút, còn khoảng 1.1 phút


In [16]:
main(["freeze", "--note", "lý do chọn lớp của nhóm"])

Đã chốt 100 lớp -> class_selection/classes.txt
  Theo vai trò: {'confusable': 76, 'control': 21, 'included': 3}
  Theo bậc khó: {'easy': 37, 'hard': 35, 'medium': 28}
  Mã kiểm tra: eea8a83a231a51b9… (ghi vào báo cáo / README)

Bước tiếp: lấy dữ liệu cho các lớp đã chốt, ví dụ:
  python select_subset.py --source kaggle --root <ROOT> --class-file class_selection/classes.txt --n-per-class 20000


In [17]:
"""
Chọn tập con Quick, Draw! theo ý mình: số lớp, lớp cụ thể, số hình mỗi lớp.

Hỗ trợ 2 nguồn:
  - kaggle : dữ liệu cuộc thi "Quick, Draw! Doodle Recognition" (train_simplified/*.csv)
  - gcs    : bucket gốc của Google (simplified/*.ndjson), tải từng lớp một

Ví dụ:
  # Kaggle, chọn 50 lớp ngẫu nhiên, 20K hình/lớp
  python select_subset.py --source kaggle \
      --root /kaggle/input/quickdraw-doodle-recognition \
      --n-classes 50 --n-per-class 20000

  # Chỉ định lớp cụ thể (dùng file txt, mỗi dòng một lớp)
  python select_subset.py --source kaggle --root ... \
      --class-file my_classes.txt --n-per-class 30000

  # Tải từ Google, chỉ vài lớp
  python select_subset.py --source gcs --classes "cat,dog,tiger,lion,bear" \
      --n-per-class 10000

  # Chỉ xem danh sách lớp rồi thoát
  python select_subset.py --source kaggle --root ... --list
"""

import argparse
import json
import os
import random
import sys
import urllib.parse
import urllib.request

import pandas as pd

GCS_URL = "https://storage.googleapis.com/quickdraw_dataset/full/simplified/{}.ndjson"
GCS_CATEGORIES = ("https://raw.githubusercontent.com/googlecreativelab/"
                  "quickdraw-dataset/master/categories.txt")
COLS = ["key_id", "word", "countrycode", "recognized", "drawing"]


# ---------- liệt kê lớp ----------
def list_classes(source, root):
    if source == "kaggle":
        folder = os.path.join(root, "train_simplified")
        return sorted(f[:-4] for f in os.listdir(folder) if f.endswith(".csv"))
    with urllib.request.urlopen(GCS_CATEGORIES) as r:
        return sorted(line.strip() for line in r.read().decode().splitlines() if line.strip())


# ---------- đọc một lớp ----------
def load_class(source, root, name, cache_dir):
    if source == "kaggle":
        path = os.path.join(root, "train_simplified", f"{name}.csv")
        df = pd.read_csv(path, usecols=COLS)
        df["recognized"] = df["recognized"].astype(str).str.lower() == "true"
        return df

    # gcs: tải file ndjson của lớp (cache lại để khỏi tải lần 2)
    os.makedirs(cache_dir, exist_ok=True)
    local = os.path.join(cache_dir, f"{name}.ndjson")
    if not os.path.exists(local):
        url = GCS_URL.format(urllib.parse.quote(name))
        print(f"  tải {url}")
        urllib.request.urlretrieve(url, local)
    rows = []
    with open(local) as f:
        for line in f:
            d = json.loads(line)
            rows.append({
                "key_id": int(d["key_id"]),
                "word": d["word"],
                "countrycode": d.get("countrycode", ""),
                "recognized": bool(d["recognized"]),
                "drawing": json.dumps(d["drawing"], separators=(",", ":")),
            })
    return pd.DataFrame(rows)


# ---------- chọn mẫu ----------
def sample_class(df, n, seed, only_recognized):
    before = len(df)
    df = df.drop_duplicates("key_id").drop_duplicates("drawing")  # bỏ trùng
    if only_recognized:
        df = df[df["recognized"]]
    if len(df) < n:
        print(f"  ! chỉ có {len(df)} hình (cần {n}), lấy hết")
        n = len(df)
    out = df.sample(n=n, random_state=seed)  # lấy ngẫu nhiên, không lấy N dòng đầu
    print(f"  {before:>7} hình -> sau lọc {len(df):>7} -> lấy {n}")
    return out


def sub(argv=None):
    # Chạy trong notebook mà không truyền tham số -> hướng dẫn thay vì crash
    if argv is None and "ipykernel" in sys.modules:
        print('Trong notebook, gọi kèm tham số, ví dụ:\n'
              'main(["--source", "kaggle", "--root", "/kaggle/input/quickdraw-doodle-recognition",\n'
              '      "--n-classes", "50", "--n-per-class", "20000"])')
        return
    p = argparse.ArgumentParser()
    p.add_argument("--source", choices=["kaggle", "gcs"], required=True)
    p.add_argument("--root", default=".", help="thư mục dữ liệu Kaggle")
    p.add_argument("--list", action="store_true", help="chỉ in danh sách lớp")
    g = p.add_mutually_exclusive_group()
    g.add_argument("--classes", help='danh sách lớp, vd "cat,dog,lion"')
    g.add_argument("--class-file", help="file txt, mỗi dòng một tên lớp")
    g.add_argument("--n-classes", type=int, help="số lớp chọn ngẫu nhiên")
    p.add_argument("--n-per-class", type=int, default=20000)
    p.add_argument("--only-recognized", action="store_true",
                   help="chỉ giữ hình game đã nhận ra (sẽ làm lệch về hình dễ!)")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--out", default="quickdraw_subset")
    p.add_argument("--cache-dir", default="gcs_cache")
    a = p.parse_args(argv)

    all_classes = list_classes(a.source, a.root)
    if a.list:
        print(f"{len(all_classes)} lớp:")
        print("\n".join(all_classes))
        return

    # xác định lớp cần lấy
    if a.classes:
        classes = [c.strip() for c in a.classes.split(",") if c.strip()]
    elif a.class_file:
        with open(a.class_file) as f:
            classes = [l.strip() for l in f if l.strip()]
    elif a.n_classes:
        classes = sorted(random.Random(a.seed).sample(all_classes, a.n_classes))
    else:
        sys.exit("Cần --classes, --class-file hoặc --n-classes")

    missing = [c for c in classes if c not in all_classes]
    if missing:
        sys.exit(f"Không có các lớp: {missing}. Chạy --list để xem tên đúng.")

    print(f"Lấy {len(classes)} lớp x {a.n_per_class} hình, seed={a.seed}")
    parts = []
    for i, c in enumerate(classes, 1):
        print(f"[{i}/{len(classes)}] {c}")
        df = load_class(a.source, a.root, c, a.cache_dir)
        parts.append(sample_class(df, a.n_per_class, a.seed, a.only_recognized))

    data = pd.concat(parts, ignore_index=True)
    data["label"] = data["word"].map({c: i for i, c in enumerate(classes)})

    # lưu dữ liệu + danh sách lớp (để tái lập kết quả)
    try:
        data.to_parquet(f"{a.out}.parquet", index=False)
        path = f"{a.out}.parquet"
    except ImportError:
        data.to_csv(f"{a.out}.csv.gz", index=False)
        path = f"{a.out}.csv.gz"
    with open(f"{a.out}_classes.txt", "w") as f:
        f.write("\n".join(classes))

    print(f"\nĐã lưu {len(data)} hình vào {path}")
    print(f"Danh sách lớp: {a.out}_classes.txt")
    print(f"Số quốc gia: {data['countrycode'].nunique()}, "
          f"tỉ lệ recognized: {data['recognized'].mean():.1%}")

In [18]:
sub(["--source", "kaggle",
     "--root", "/kaggle/input/competitions/quickdraw-doodle-recognition",
     "--class-file", "class_selection/classes.txt",
     "--n-per-class", "30000",
     "--seed", "42",
     "--out", "quickdraw_100c_20k"])

Lấy 100 lớp x 30000 hình, seed=42
[1/100] The Mona Lisa
   121383 hình -> sau lọc  121383 -> lấy 30000
[2/100] alarm clock
   123399 hình -> sau lọc  123394 -> lấy 30000
[3/100] angel
   149736 hình -> sau lọc  149725 -> lấy 30000
[4/100] ant
   124612 hình -> sau lọc  124604 -> lấy 30000
[5/100] anvil
   126231 hình -> sau lọc  126221 -> lấy 30000
[6/100] apple
   144722 hình -> sau lọc  144721 -> lấy 30000
[7/100] basketball
   133793 hình -> sau lọc  133791 -> lấy 30000
[8/100] bat
   118114 hình -> sau lọc  118106 -> lấy 30000
[9/100] bathtub
   174336 hình -> sau lọc  174310 -> lấy 30000
[10/100] bird
   133572 hình -> sau lọc  133557 -> lấy 30000
[11/100] blackberry
   128153 hình -> sau lọc  128139 -> lấy 30000
[12/100] boomerang
   142682 hình -> sau lọc  142678 -> lấy 30000
[13/100] bowtie
   130283 hình -> sau lọc  130281 -> lấy 30000
[14/100] bracelet
   119416 hình -> sau lọc  119414 -> lấy 30000
[15/100] butterfly
   117999 hình -> sau lọc  117998 -> lấy 30000
[16/100] cam